In [ ]:
import os
import random
import warnings
from datetime import datetime
from rich import print

import anndata as ad
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import squidpy as sq
from matplotlib import gridspec
from sklearn.preprocessing import MinMaxScaler

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                create_new_color_dict,
                                compute_communication_gp_network,
                                visualize_communication_gp_network,
                                extract_gp_dict_from_mebocost_ms_interactions,
                                extract_gp_dict_from_nichenet_lrt_interactions,
                                extract_gp_dict_from_omnipath_lr_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                generate_enriched_gp_info_plots)

PATH = "/home/kibr/Work/Vascular_atlas"
import torch
## set path and parameters
os.chdir(PATH)

In [ ]:
### Dataset ###
dataset = "xenium"
species = "human"
spatial_key = "spatial"
batches = ["HIP","IFG","Pons"]
n_neighbors = 4
data_path = "/home/kibr/Work/Vascular_atlas/Data/Xenium/adata/"

### Model ###
# AnnData keys
counts_key = "counts"
adj_key = "spatial_connectivities"
cat_covariates_keys = ["Brain_region"]
gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"

# Architecture
cat_covariates_embeds_injection = ["gene_expr_decoder"]
cat_covariates_embeds_nums = [3]
cat_covariates_no_edges = [True]
conv_layer_encoder = "gcnconv" # change to "gatv2conv" if enough compute and memory
active_gp_thresh_ratio = 0.01

# Trainer
n_epochs = 400
n_epochs_all_gps = 25
lr = 0.001
lambda_edge_recon = 500000.
lambda_gene_expr_recon = 300.
lambda_l1_masked = 0. # prior GP  regularization
lambda_l1_addon = 30. # de novo GP regularization
edge_batch_size = 4096 # increase if more memory available or decrease to save memory
n_sampled_neighbors = 4
use_cuda_if_available = True

### Analysis ###
cell_type_key = "cell_type"
latent_leiden_resolution = 0.2
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "Brain_region"
spot_size = 40
differential_gp_test_results_key = "nichecompass_differential_gp_test_results"

In [ ]:
adata = sc.read_h5ad(f"{PATH}/Results/Revision_2/Xenium/adata_with_niches.h5ad")
print(adata)
## change the cell type label for "Mixed" to "Unannotated"
# adata.obs[cell_type_key] = adata.obs[cell_type_key].replace({"Mixed": "Unannotated"})

In [ ]:
## plot the latent clusters using niche colors
latent_cluster_colors = create_new_color_dict(
    adata=adata,
    cat_key=latent_cluster_key)

# groups = None
sc.pl.spatial(adata=adata[adata.obs[sample_key] == "IFG"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"IFG"})",
              return_fig = bool,
            #   save=f"_IFG_niche_spatial.svg",
              show=True)
# # Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_IFG.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

# groups = None
sc.pl.spatial(adata=adata[ adata.obs[sample_key] == "HIP"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"HIP"})",
              return_fig = bool,
            #   save=f"_HIP_niche_spatial.svg",
              show=True)
# Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_HIP.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

# groups = None
sc.pl.spatial(adata=adata[adata.obs[sample_key] == "Pons"],
              color=[latent_cluster_key],
              # groups=groups,
              palette=latent_cluster_colors,
              spot_size=spot_size,
              title=f"Batches in Physical Space \n"
                    f"(Sample: {"Pons"})",
            #   save=f"_Pons_niche_spatial.svg",    
              show=True)
# Save as rasterized PDF
# plt.savefig("./Results/Revision_2/Xenium/NicheCompass/umap_niches_Pons.pdf", dpi=300, format="pdf", bbox_inches="tight")
# plt.show() 

In [ ]:
cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",  
    "Unannotated": "#7f7f7f",             
    "Neuron": "#08415C",            
    "Microglia": "#d62728",         
    "Astrocyte": "#F06719",         
    "OPC": "#0072B2",              
    # "Capillary1": "#A6DBD0",        
    "Capillary": "#66C2A5",        
    "Venous": "#4393C3",           
    "Pericyte": "#e377c2",          
    "SMC": "#9467bd",               
    "Arterial": "#fcbe05",         
    "Fibroblast": "#5b844d"         
}

ordered_cell_types = [
    "Unannotated",
    "Oligodendrocytes",
    "OPC",
    "Neuron",
    "Microglia",
    "Astrocyte",      
    "Pericyte",
    "SMC",
    "Fibroblast",
    "Arterial",
    "Venous",
    "Capillary",
]

In [ ]:
## setting the saving path for the figurtes
sc.set_figure_params(figsize=(10, 10), frameon=False)
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium/"

samples = adata.obs[sample_key].unique().tolist()

for sample in samples:
    adata_batch = adata[adata.obs[sample_key] == sample]
    
    print(f"Summary of sample {sample}:")
    print(f"Number of nodes (observations): {adata_batch.layers[counts_key].shape[0]}")
    print(f"Number of node features (genes): {adata_batch.layers[counts_key].shape[1]}")

    # Visualize cell-level annotated data in physical space
    sc.pl.spatial(adata_batch,
                          color=cell_type_key,
                          palette=cell_type_colors,
                          spot_size=spot_size,
                          save = f"_{sample}_cell_type_spatial.svg")  

## Highlight the DEG for selected cell types between niches (regions)

In [ ]:
adata.obs['cell_type'].value_counts()
# adata.obs['Brain_region'].value_counts()

### Neurons first

In [ ]:
## subset the capillary and check their expression across the niches
# adata_sub = adata[(adata.obs['cell_type'].isin(["Neuron"])) & (adata.obs['Brain_region'].isin(["Pons", "HIP"])) & (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()
adata_sub = adata[(adata.obs['cell_type'].isin(["Neuron"]))& (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()

print(adata_sub)  

In [ ]:
## find the differentially expressed genes across the niches in the capillary cells
sc.tl.rank_genes_groups(adata_sub, groupby=latent_cluster_key, method='wilcoxon')
sc.pl.rank_genes_groups(adata_sub, n_genes=20, sharey=False)
plt.show()


In [ ]:
## draw the dot plot based on the selected genes from the DE test
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium"
# genes_oi = {"Pons": ["MFSD2A","SLC7A5","SLCO2B1","SLC2A1","NOTCH1"],
#             "HIP": ["B2M","PECAM1","CALM1","CLU","VWF","SLC1A2",'SLC1A3'],}
genes_oi = {"Pons": ["NRG1","PVALB",'SLC6A1'],
            "HIP": ["HCN1","RORB",'RBFOX1'],}
sc.pl.dotplot(adata_sub, var_names=genes_oi, groupby=latent_cluster_key,standard_scale='var',
              save=f"Neuron_niches_deg.svg")

### Endothelial cells

In [ ]:
## subset the capillary and check their expression across the niches
# adata_sub = adata[(adata.obs['cell_type'].isin(["Neuron"])) & (adata.obs['Brain_region'].isin(["Pons", "HIP"])) & (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()
adata_sub = adata[(adata.obs['cell_type'].isin(["Arterial","Capillary","Venous"]))& (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()

print(adata_sub)  

In [ ]:
## draw the dot plot based on the selected genes from the DE test
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium"
# genes_oi = {"Pons": ["MFSD2A","SLC7A5","SLCO2B1","SLC2A1","NOTCH1"],
#             "HIP": ["B2M","PECAM1","CALM1","CLU","VWF","SLC1A2",'SLC1A3'],}
genes_oi = {"Pons": ["MFSD2A","SLC2A1",'NOTCH1',"SLC7A5"],
            "HIP": ["B2M","PECAM1",'CALM1',"CLU"],}
sc.pl.dotplot(adata_sub, var_names=genes_oi, groupby=latent_cluster_key,standard_scale='var',
              save=f"Endothelial_niches_deg.svg")

## Astrocytes

In [ ]:
## subset the capillary and check their expression across the niches
# adata_sub = adata[(adata.obs['cell_type'].isin(["Neuron"])) & (adata.obs['Brain_region'].isin(["Pons", "HIP"])) & (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()
adata_sub = adata[(adata.obs['cell_type'].isin(["Astrocyte"]))& (adata.obs[latent_cluster_key].isin(["0","2","5"]))].copy()

print(adata_sub)  

In [ ]:
## find the differentially expressed genes across the niches in the capillary cells
sc.tl.rank_genes_groups(adata_sub, groupby=latent_cluster_key, method='wilcoxon')
sc.pl.rank_genes_groups(adata_sub, n_genes=20, sharey=False)
plt.show()


In [ ]:
## output the top 50 genes of each niche in astrocytes into a list
genes_oi = {}
for cluster in adata_sub.obs[latent_cluster_key].cat.categories:
    genes_oi[cluster] = adata_sub.uns['rank_genes_groups']['names'][cluster][:50].tolist()

genes_oi

In [ ]:
## draw the dot plot based on the selected genes from the DE test
sc.settings.figdir = f"{PATH}/Results/Revision_2/Xenium"
# genes_oi = {"Pons": ["MFSD2A","SLC7A5","SLCO2B1","SLC2A1","NOTCH1"],
#             "HIP": ["B2M","PECAM1","CALM1","CLU","VWF","SLC1A2",'SLC1A3'],}
genes_oi = {"Pons": ["BAG3","CLU",'NOTCH1'],
            "HIP": ["SLC1A2","SLC4A4",'APOE'],}
sc.pl.dotplot(adata_sub, var_names=genes_oi, groupby=latent_cluster_key,standard_scale='var',
              save=f"Astrocyte_niches_deg.svg")

In [ ]:
adata_sub = adata[adata.obs[sample_key] == "HIP"].copy()
print(adata_sub)

In [ ]:
adata.obs["sample"] = adata.obs.index.str.rsplit("_", n=1).str[0]  # e.g. 'HIP'
# adata.obs["cell_id"] = adata.obs.index.str.rsplit("_", n=1).str[1]  # e.g. '0', '2', '3'

print(adata.obs[["sample", "cell_id"]].head())
print(adata.obs["sample"].unique())  # check how many samples

In [ ]:
for sample in adata.obs["sample"].unique():
    mask = adata.obs["sample"] == sample
    df = pd.DataFrame({
        "cell_id": adata.obs.loc[mask, "cell_id"],
        "group": adata.obs.loc[mask, latent_cluster_key],   # your niche column
        "cell_type": adata.obs.loc[mask, 'cell_type']
            # optional: add cell type info if available
    })
    df.to_csv(f"{sample}_niche_assignments.csv", index=False)
    print(f"Saved {sample}_niche_assignments.csv — {len(df)} cells")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# Publication-quality rcParams
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 7,
    'axes.titlesize': 8,
    'axes.titleweight': 'normal',
    'legend.fontsize': 6.5,
    'legend.title_fontsize': 7,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

cell_type_colors = {
    "Oligodendrocytes": "#00BFC4",
    "Unannotated": "#7f7f7f",
    "Neuron": "#08415C",
    "Microglia": "#d62728",
    "Astrocyte": "#F06719",
    "OPC": "#0072B2",
    "Capillary": "#66C2A5",
    "Venous": "#4393C3",
    "Pericyte": "#e377c2",
    "SMC": "#9467bd",
    "Arterial": "#fcbe05",
    "Fibroblast": "#5b844d",
}

# Compute composition
niches = adata.obs[latent_cluster_key].unique()
cell_types = adata.obs['cell_type'].unique()
composition = pd.DataFrame(0, index=niches, columns=cell_types)
for niche in niches:
    niche_mask = adata.obs[latent_cluster_key] == niche
    for cell_type in cell_types:
        cell_type_mask = adata.obs['cell_type'] == cell_type
        composition.loc[niche, cell_type] = (niche_mask & cell_type_mask).sum()


def plot_niche_pie(values, labels, colors, title,
                   deemphasize=('Unannotated',),
                   min_pct_label=5.0, save_path=None):
    """Publication-style pie chart with seamless wedges and de-emphasized categories."""
    values = np.asarray(values, dtype=float)
    total = values.sum()
    pcts = 100 * values / total

    def autopct_fn(pct):
        return f'{pct:.0f}%' if pct >= min_pct_label else ''

    fig, ax = plt.subplots(figsize=(3.2, 2.4), dpi=300)
    fig.patch.set_facecolor('white')

    # Match wedge edge color to its fill so adjacent wedges blend seamlessly
    wedges, _, autotexts = ax.pie(
        values,
        colors=colors,
        autopct=autopct_fn,
        pctdistance=0.7,
        startangle=90,
        counterclock=False,
        wedgeprops=dict(linewidth=0),
        textprops=dict(fontsize=10, color='black'),
    )
    for w, c in zip(wedges, colors):
        w.set_edgecolor(c)

    # De-emphasize specified categories (e.g., 'Unannotated')
    for w, lbl, t in zip(wedges, labels, autotexts):
        if lbl in deemphasize:
            w.set_alpha(0.35)
            t.set_text('')  # hide its percentage label too

    ax.set_title(title, pad=6)

    # Legend: sorted by abundance, exclude zero-count types
    order = np.argsort(-values)
    legend_labels, legend_handles = [], []
    for i in order:
        if values[i] == 0:
            continue
        is_dim = labels[i] in deemphasize
        alpha = 0.35 if is_dim else 1.0
        legend_labels.append(f'{labels[i]} ({pcts[i]:.1f}%)')
        legend_handles.append(
            mpl.patches.Patch(facecolor=colors[i], edgecolor='none', alpha=alpha)
        )

    ax.legend(
        legend_handles, legend_labels,
        loc='center left', bbox_to_anchor=(1.0, 0.5),
        frameon=False, handlelength=1.0, handleheight=1.0,
        labelspacing=0.4, borderaxespad=0,
    )

    ax.set(aspect='equal')
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)
    plt.show()


for niche in niches:
    values = composition.loc[niche].values
    labels = composition.columns.tolist()
    colors = [cell_type_colors[ct] for ct in labels]
    plot_niche_pie(
        values, labels, colors,
        title=f'Niche {niche}',
        save_path=f'./Results/Revision_2/Xenium/niche_{niche}_composition.svg',
    )

## Final proportions plot

In [ ]:
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'

## Plot the niche composition of cell types in selected brain regions
selected_regions = ["HIP", "IFG","Pons"]

df_counts = (adata.obs.groupby(["Brain_region", latent_cluster_key])
             .size().unstack())
## only keep the region of interest
df_counts = df_counts[df_counts.index.isin(selected_regions)]
df_counts = df_counts.div(df_counts.sum(axis=1), axis=0)
niches = df_counts.columns.tolist()
niche_colors = {niche: latent_cluster_colors[niche] for niche in niches}

ax = df_counts.plot(kind="bar", stacked=True, figsize=(4,6),
                    color=[niche_colors[niche] for niche in niches])

# Add proportion text on each segment
for i, region in enumerate(df_counts.index):
    cumulative = 0
    for niche in niches:
        proportion = df_counts.loc[region, niche]
        if proportion > 0:  # only label non-zero segments
            ax.text(i, cumulative + proportion / 2,
                    f"{proportion:.2%}",
                    ha="center", va="center",
                    fontsize=9, color="black")
        cumulative += proportion

# Remove y-axis
ax.yaxis.set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

legend = plt.legend(bbox_to_anchor=(1, 1), loc="upper left", prop={'size': 10})
legend.set_title("Cell Type Annotations", prop={'size': 10})

plt.title("Niches Composition of Brain Regions")
plt.xlabel("Brain Region")
plt.grid(False)

file_path = f"{PATH}/Results/Revision_2/Xenium/region_composition_niches_{'_'.join(selected_regions)}.svg"
# plt.savefig(file_path, bbox_extra_artists=(legend,), bbox_inches="tight")
plt.show()